# Fig. 6 — shape-index time curves (right-hand plots, $f_{mut}$ = 0.05 and 0.25)

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors
from matplotlib.cm import ScalarMappable
import numpy as np
import pyprojroot.here as here
import pandas as pd
from vertex_lite.custom import load_data_v3 as load_data

In [ ]:
def filter_data_shape_index(time=None, local=False, cluster=False):
    rpath = here('results/resultados_cluster' if cluster else 'results/resultados_random')

    value_column = 'shape_neigh' if local else 'shape_global'
    filename = 'mutant.parquet' if local else 'global.parquet'
    
    required_cols = ['time', value_column]
    dfs = []

    for pct in [0.05, 0.25]:
        data = load_data(
            f'{rpath}/noise_*/pct_{pct}/sim_*/{filename}',
            columns=required_cols,
            time_filter=time
        )
        if data is not None and not data.empty:
            dfs.append(data)

    if not dfs:
        print("No data found")
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True).dropna(subset=[value_column])

    if df.empty:
        print("No shape-index data found")
        return pd.DataFrame()

    # Mean over cells within each simulation
    per_sim = (
        df.groupby(['time', 'mesh_type', 'pct_mutant', 'sim_id'])[value_column]
        .mean()
        .reset_index(name='shape_index')
    )

    return per_sim


def summarize_shape_index(per_sim_df):
    if per_sim_df.empty:
        return pd.DataFrame()

    stats = (
        per_sim_df.groupby(["time", "mesh_type", "pct_mutant"])["shape_index"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    stats["sem"] = stats["std"] / np.sqrt(stats["count"])
    return stats


def plot_si_time(stats, local=False, cluster=False,max_time=5000):
    sns.set_theme(
        style="white",
        context="paper",
        rc={
            "axes.edgecolor": "black",
            "axes.linewidth": 1.0,
            "axes.facecolor": "white",
            "figure.facecolor": "white",
        }
    )
    savedir = here("results/si/plots/si_time")
    os.makedirs(savedir, exist_ok=True)

    local_label = "local" if local else "global"
    cluster_label = "cluster" if cluster else "uniform"

    for pct in sorted(stats["pct_mutant"].unique()):
        d0 = stats[stats["pct_mutant"] == pct].copy()
        if d0.empty:
            continue

        fig, ax = plt.subplots(figsize=(6.5, 4))

        meshes_sorted = sorted(d0["mesh_type"].unique())
        norm = colors.Normalize(vmin=min(meshes_sorted), vmax=max(meshes_sorted))
        cmap = sns.light_palette("#2B352A", as_cmap=True)

        for mesh in meshes_sorted:
            d = d0[d0["mesh_type"] == mesh].sort_values("time")
            d = d[d["time"] <= max_time]

            x = d["time"].to_numpy()
            y = d["mean"].to_numpy()
            sem = d["sem"].to_numpy()
            color = cmap(norm(mesh))

            ax.fill_between(
                x, y - sem, y + sem,
                color=color, alpha=0.18, linewidth=0
            )
            ax.plot(
                x, y,
                color=color, linewidth=2.25
            )

        ax.set_title(
            f"Mutant percentage {pct} | {local_label} | {cluster_label}",
            fontsize=14
        )
        ax.set_xlabel("Time step", fontsize=16)
        ax.set_ylabel("Shape index", fontsize=16)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
   
        ax.set_xticks(range(0, max_time + 1, 1000))

        ax.tick_params(
            axis="both", which="both", direction="out",
            length=7, labelsize=18, width=1, bottom=True, left=True
        )
        ax.tick_params(axis='x', rotation=45)

        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.05)
        cbar.set_label("Initial mesh noise", fontsize=14)
        cbar.ax.tick_params(labelsize=12)
        cbar.outline.set_visible(False)
        plt.ylim(3.7, 4.05)
        plt.tight_layout()
        fig.savefig(
            f"{savedir}/si_pct_{pct}_{local_label}_{cluster_label}_{max_time}.svg",
            format="svg"
        )
        plt.close(fig)

In [ ]:
for local in [True, False]:
    for cluster in [True, False]:
        df = filter_data_shape_index(local=local, cluster=cluster)
        stats = summarize_shape_index(df)
        plot_si_time(stats, local=local, cluster=cluster, max_time=5000)